# **NLP-Based Resume Keyword Extraction and Job Description Matching Using TF-IDF**

In [ ]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Dataset Loading and Exploration

In [ ]:
import zipfile
import os

zip_path = "/content/archive.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/resume_dataset")

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
for root, dirs, files in os.walk("/content/resume_dataset"):
    for file in files:
        if file == "training_data.csv":
            print(os.path.join(root, file))

/content/resume_dataset/training_data.csv


In [ ]:
df = pd.read_csv("/content/resume_dataset/training_data.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (10000, 7)

Columns:
['Resume ID', 'Resume Text', 'Education', 'Experience Years', 'Skills', 'Job Role', 'Category']


In [ ]:
df.head()

,Resume ID,Resume Text,Education,Experience Years,Skills,Job Role,Category
0,R000000,Education: Bachelor's in Computer Science Expe...,Bachelor's in Computer Science,2,Market Research|Maven|Java|REST API|Spring Boo...,Java Backend Developer,Technology
1,R000001,Education: Master's in Microbiology Experience...,Master's in Microbiology,3,Agile|Data Analysis|Precision|Problem Solving|...,Microbiologist,Science & Research
2,R000002,Education: Apprenticeship Experience: 2 years ...,Apprenticeship,2,React|Customer Service|Technical Knowledge|Plu...,Plumber,Skilled Trades
3,R000003,Education: Bachelor's in Computer Science Expe...,Bachelor's in Computer Science,0,MySQL|Query Optimization|Symfony|Terraform|PHP...,PHP Developer,Technology
4,R000004,Education: Bachelor's in Design Experience: 3 ...,Bachelor's in Design,3,Problem Solving|Docker|Creativity|Scrum|Ansibl...,Art Director,Creative & Design


In [ ]:
print("Dataset Information:")
df.info()

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Resume ID         10000 non-null  object
 1   Resume Text       10000 non-null  object
 2   Education         10000 non-null  object
 3   Experience Years  10000 non-null  int64 
 4   Skills            10000 non-null  object
 5   Job Role          10000 non-null  object
 6   Category          10000 non-null  object
dtypes: int64(1), object(6)
memory usage: 547.0+ KB


In [ ]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
Resume ID           0
Resume Text         0
Education           0
Experience Years    0
Skills              0
Job Role            0
Category            0
dtype: int64


In [ ]:
print("Number of unique values:")
print(df.nunique())

Number of unique values:
Resume ID           10000
Resume Text         10000
Education             257
Experience Years       14
Skills              10000
Job Role              324
Category               42
dtype: int64


## 2. Data Preprocessing

In [ ]:
# Select the resume text column
resume_column = "Resume Text"

# Fill missing values
df[resume_column] = df[resume_column].fillna("")

# Convert to lowercase
df[resume_column] = df[resume_column].str.lower()

# Remove punctuation and special characters
df[resume_column] = df[resume_column].apply(
    lambda x: re.sub(r'[^a-zA-Z0-9\s]', ' ', x)
)

# Remove extra spaces
df[resume_column] = df[resume_column].apply(
    lambda x: re.sub(r'\s+', ' ', x).strip()
)

df[[resume_column]].head()

,Resume Text
0,education bachelor s in computer science exper...
1,education master s in microbiology experience ...
2,education apprenticeship experience 2 years sk...
3,education bachelor s in computer science exper...
4,education bachelor s in design experience 3 ye...


## 3. TF-IDF Feature Extraction

In [ ]:
# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

# Convert resume text into TF-IDF features
tfidf_matrix = tfidf.fit_transform(df[resume_column])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (10000, 783)


In [ ]:
# Get all words from the TF-IDF vocabulary
feature_names = tfidf.get_feature_names_out()

# Extract top keywords from each resume
def extract_keywords(row, n=10):
    scores = row.toarray().flatten()
    top_indices = scores.argsort()[-n:][::-1]
    return [feature_names[i] for i in top_indices if scores[i] > 0]

df["Keywords"] = [extract_keywords(tfidf_matrix[i]) for i in range(len(df))]

df[["Resume Text", "Keywords"]].head()

,Resume Text,Keywords
0,education bachelor s in computer science exper...,"[boot, spring, maven, microservices, statistic..."
1,education master s in microbiology experience ...,"[microbiology, scala, agile, precision, develo..."
2,education apprenticeship experience 2 years sk...,"[plumbing, apprenticeship, code, knowledge, po..."
3,education bachelor s in computer science exper...,"[symfony, terraform, query, php, mysql, optimi..."
4,education bachelor s in design experience 3 ye...,"[principles, design, ansible, team, scrum, doc..."


## 5. Job Description Matching

In [ ]:
# Example job description
job_description = """
Looking for a Python developer with skills in Python, SQL, Django,
machine learning, data analysis and Git.
"""

# Preprocess the job description
job_description = job_description.lower()
job_description = re.sub(r'[^a-zA-Z0-9\s]', ' ', job_description)
job_description = re.sub(r'\s+', ' ', job_description).strip()

# Transform job description using the same TF-IDF vectorizer
job_vector = tfidf.transform([job_description])

# Calculate similarity with all resumes
similarity_scores = cosine_similarity(job_vector, tfidf_matrix).flatten()

# Add similarity scores to the dataset
df["Match Score"] = similarity_scores * 100

# Display top matching resumes
df[["Resume ID", "Job Role", "Keywords", "Match Score"]].sort_values(
    by="Match Score", ascending=False
).head(5)

,Resume ID,Job Role,Keywords,Match Score
7150,R007150,Data Scientist,"[data, django, machine, prototyping, ux, ui, v...",57.118775
9399,R009399,Data Scientist,"[data, machine, tensorflow, kotlin, statistics...",55.545165
5373,R005373,Software Engineer,"[git, software, machine, java, making, decisio...",54.991393
6092,R006092,Data Scientist,"[data, agile, machine, tensorflow, statistics,...",54.770573
5284,R005284,Data Scientist,"[data, query, machine, tensorflow, statistics,...",52.718625


## 6. Evaluation

In [ ]:
# Display matching results with their similarity scores

evaluation_results = df[["Resume ID", "Job Role", "Match Score"]].sort_values(
    by="Match Score", ascending=False
).head(10)

evaluation_results

,Resume ID,Job Role,Match Score
7150,R007150,Data Scientist,57.118775
9399,R009399,Data Scientist,55.545165
5373,R005373,Software Engineer,54.991393
6092,R006092,Data Scientist,54.770573
5284,R005284,Data Scientist,52.718625
6582,R006582,Data Scientist,52.284111
4709,R004709,Data Scientist,50.872147
6395,R006395,Data Scientist,50.702500
8241,R008241,Data Scientist,50.301838
6977,R006977,Jupyter Developer,50.183253


In [ ]:
# Basic statistics of the match scores

print("Average Match Score:", round(df["Match Score"].mean(), 2), "%")
print("Highest Match Score:", round(df["Match Score"].max(), 2), "%")
print("Lowest Match Score:", round(df["Match Score"].min(), 2), "%")

Average Match Score: 4.3 %
Highest Match Score: 57.12 %
Lowest Match Score: 0.3 %


## 7. Results

In [ ]:
# Display the top 5 matching resumes

top_matches = df[["Resume ID", "Job Role", "Match Score"]].sort_values(
    by="Match Score", ascending=False
).head(5)

top_matches

,Resume ID,Job Role,Match Score
7150,R007150,Data Scientist,57.118775
9399,R009399,Data Scientist,55.545165
5373,R005373,Software Engineer,54.991393
6092,R006092,Data Scientist,54.770573
5284,R005284,Data Scientist,52.718625


In [ ]:
print("Best Match:")
print("Resume ID:", top_matches.iloc[0]["Resume ID"])
print("Job Role:", top_matches.iloc[0]["Job Role"])
print("Match Score:", round(top_matches.iloc[0]["Match Score"], 2), "%")

Best Match:
Resume ID: R007150
Job Role: Data Scientist
Match Score: 57.12 %


## 8. Conclusion

This project successfully used Natural Language Processing techniques for resume keyword extraction and job description matching. TF-IDF was used to identify important terms from resumes, and Cosine Similarity was used to measure the similarity between resumes and the given job description. Among the 10,000 resumes, the highest matching score was 57.12% for Resume ID R007150, with the job role Data Scientist. The results show that the proposed approach can be used to identify resumes that are more relevant to a given job description.